# Clustering Analysis on Merchant Cost Patterns — MCC 4121 (Taxicabs & Limousines)


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor

print('All libraries loaded successfully.')


## 2. Data Loading & Feature Engineering

In [ ]:
df_train_raw    = pd.read_csv('4121_train_iso_with_costs.csv')
df_validate_raw = pd.read_csv('4121_validate_iso_with_costs.csv')
df_test_raw     = pd.read_csv('4121_test_iso_with_costs.csv')

print('Raw transaction shapes:')
print(f'  Train:    {df_train_raw.shape}')
print(f'  Validate: {df_validate_raw.shape}')
print(f'  Test:     {df_test_raw.shape}')
print(f'\nColumns: {df_train_raw.columns.tolist()}')


In [ ]:
print('=== TRANSACTION-LEVEL DATA QUALITY ===')
for name, df in [('Train', df_train_raw), ('Validate', df_validate_raw), ('Test', df_test_raw)]:
    print(f'\n{name}:')
    print(f'  Rows:             {len(df):,}')
    print(f'  Unique merchants: {df["merchant_id"].nunique():,}')
    print(f'  NaN proc_cost:    {df["proc_cost"].isna().sum():,}  ({df["proc_cost"].isna().mean()*100:.1f}%)')
    print(f'  Card brands:      {df["card_brand"].value_counts().to_dict()}')
    print(f'  Card types:       {df["card_type"].value_counts().to_dict()}')


In [ ]:
def engineer_merchant_features(df):
    """
    Aggregate raw transaction data to one row per merchant with enriched features.

    Features engineered:
      cost_percent       - Mean proc_cost/amount (core cost efficiency metric)
      cost_percent_stdev - Std of proc_cost/amount (cost variability)
      avg_txn_amount     - Mean transaction size
      txn_count          - Transaction volume
      chip_rate          - Fraction of chip (lower-risk EMV) transactions
      visa_rate          - Fraction of Visa-branded transactions
      credit_rate        - Fraction of credit card transactions
      fraud_rate         - Fraction of flagged fraud transactions

    Note: Rows where proc_cost is NaN (e.g. Amex with no rate configured)
    are excluded from cost features only; volume/mix features use all rows.
    """
    df = df.copy()

    df_costed = df.dropna(subset=['proc_cost', 'amount'])
    df_costed = df_costed[df_costed['amount'] > 0].copy()
    df_costed['cost_pct'] = df_costed['proc_cost'] / df_costed['amount']

    cost_agg = df_costed.groupby('merchant_id').agg(
        cost_percent=('cost_pct', 'mean'),
        cost_percent_stdev=('cost_pct', 'std'),
        avg_txn_amount=('amount', 'mean'),
    )

    df['is_chip']   = (df['use_chip'] == 'Chip Transaction').astype(int)
    df['is_visa']   = (df['card_brand'].str.lower() == 'visa').astype(int)
    df['is_credit'] = (df['card_type'].str.lower() == 'credit').astype(int)
    df['is_fraud']  = pd.to_numeric(df['is_fraud'], errors='coerce').fillna(0).astype(int)

    mix_agg = df.groupby('merchant_id').agg(
        txn_count=('transaction_id', 'count'),
        chip_rate=('is_chip', 'mean'),
        visa_rate=('is_visa', 'mean'),
        credit_rate=('is_credit', 'mean'),
        fraud_rate=('is_fraud', 'mean'),
    )

    merchant_df = cost_agg.join(mix_agg, how='inner').reset_index()
    merchant_df['cost_percent_stdev'] = merchant_df['cost_percent_stdev'].fillna(0)
    return merchant_df

df_train    = engineer_merchant_features(df_train_raw)
df_validate = engineer_merchant_features(df_validate_raw)
df_test     = engineer_merchant_features(df_test_raw)

print('Merchant-level datasets created:')
print(f'  Train:    {df_train.shape}')
print(f'  Validate: {df_validate.shape}')
print(f'  Test:     {df_test.shape}')
df_train.describe()


## 3. Feature Exploration & Analysis

In [ ]:
ALL_FEATURE_COLS = [c for c in df_train.columns if c != 'merchant_id']
print(f'All feature columns ({len(ALL_FEATURE_COLS)}): {ALL_FEATURE_COLS}')

n_cols = 4
n_rows = int(np.ceil(len(ALL_FEATURE_COLS) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
fig.suptitle('Feature Distributions — MCC 4121 Merchants (Train / Validate / Test)', fontsize=14, fontweight='bold')
axes = axes.flatten()
for idx, feature in enumerate(ALL_FEATURE_COLS):
    ax = axes[idx]
    ax.hist(df_train[feature], bins=40, alpha=0.5, label='Train', density=True, color='steelblue')
    ax.hist(df_validate[feature], bins=40, alpha=0.5, label='Validate', density=True, color='orange')
    ax.hist(df_test[feature], bins=40, alpha=0.5, label='Test', density=True, color='green')
    ax.set_title(feature, fontsize=12, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
for idx in range(len(ALL_FEATURE_COLS), len(axes)):
    fig.delaxes(axes[idx])
plt.tight_layout()
plt.show()


In [ ]:
print('=' * 70)
print('CORRELATION ANALYSIS (Training Data)')
print('=' * 70)
corr = df_train[ALL_FEATURE_COLS].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix — MCC 4121 Training Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

high_corr = [(corr.columns[i], corr.columns[j], corr.iloc[i, j])
             for i in range(len(corr.columns))
             for j in range(i+1, len(corr.columns))
             if abs(corr.iloc[i, j]) > 0.8]
if high_corr:
    print('\n⚠️  Highly correlated pairs (|r| > 0.8):')
    for a, b, v in high_corr:
        print(f'  {a} <-> {b}: {v:.3f}')
else:
    print('\n✓ No feature pairs with |correlation| > 0.8')


In [ ]:
# ── IMPROVEMENT 1: Zero-variance check before VIF ──────────────────────────
print('=' * 70)
print('ZERO-VARIANCE CHECK')
print('=' * 70)
zero_var_features = [f for f in ALL_FEATURE_COLS if df_train[f].std() == 0 or df_train[f].nunique() <= 1]
near_zero_var = [f for f in ALL_FEATURE_COLS if df_train[f].std() < 1e-8]
print(f'Zero-variance features (dropped): {zero_var_features}')
print(f'Near-zero variance features: {near_zero_var}')

# fraud_rate is all-zero in training after the dataset → drop it
# Also check if chip_rate is near-constant
for f in ALL_FEATURE_COLS:
    print(f'  {f:30s}  std={df_train[f].std():.6f}  unique={df_train[f].nunique()}')

FEATURE_COLS = [f for f in ALL_FEATURE_COLS if df_train[f].std() > 1e-8 and df_train[f].nunique() > 1]
DROPPED_FEATURES = [f for f in ALL_FEATURE_COLS if f not in FEATURE_COLS]
print(f'\n✓ Features kept for clustering ({len(FEATURE_COLS)}): {FEATURE_COLS}')
if DROPPED_FEATURES:
    print(f'✗ Features dropped (zero/near-zero variance): {DROPPED_FEATURES}')


In [ ]:
print('=' * 70)
print('VARIANCE INFLATION FACTOR (VIF) — on kept features only')
print('=' * 70)
vif_data = df_train[FEATURE_COLS].values
vif_df = pd.DataFrame({
    'Feature': FEATURE_COLS,
    'VIF': [variance_inflation_factor(vif_data, i) for i in range(len(FEATURE_COLS))]
})
print(vif_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['red' if v >= 10 else 'orange' if v >= 5 else 'green' for v in vif_df['VIF']]
ax.barh(vif_df['Feature'], vif_df['VIF'], color=colors, edgecolor='black', linewidth=0.5)
ax.axvline(x=5,  color='orange', linestyle='--', linewidth=2, label='Moderate (5)')
ax.axvline(x=10, color='red',    linestyle='--', linewidth=2, label='High (10)')
ax.set_xlabel('VIF Value')
ax.set_title('VIF — Multicollinearity Check (after zero-var drop)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

high_vif = vif_df[vif_df['VIF'] >= 10]
if len(high_vif) > 0:
    print(f'\n⚠️  High VIF features (>=10): {high_vif["Feature"].tolist()}')
else:
    print('\n✓ All features have acceptable VIF (<10).')


In [ ]:
# PCA for dimensionality insight (on kept features)
print('=' * 70)
print('PCA — Dimensionality Insight')
print('=' * 70)
pca = PCA()
pca.fit(df_train[FEATURE_COLS].values)
ev     = pca.explained_variance_ratio_
cum_ev = np.cumsum(ev)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, len(ev)+1), ev, alpha=0.7, color='steelblue', label='Individual')
axes[0].step(range(1, len(cum_ev)+1), cum_ev, where='mid', color='red', label='Cumulative')
axes[0].axhline(0.90, linestyle='--', color='gray', alpha=0.7, label='90% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA Scree Plot', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(range(1, len(ev)+1))

loadings = pd.DataFrame(pca.components_.T, index=FEATURE_COLS,
                         columns=[f'PC{i+1}' for i in range(len(ev))])
sns.heatmap(loadings.iloc[:, :min(5, len(ev))], annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[1], vmin=-1, vmax=1)
axes[1].set_title('Feature Loadings on Top PCs', fontweight='bold')
plt.tight_layout()
plt.show()

n90 = np.searchsorted(cum_ev, 0.90) + 1
print(f'Components needed for 90% variance: {n90}')
for i, v in enumerate(ev):
    print(f'  PC{i+1}: {v*100:.1f}%  (cumulative: {cum_ev[i]*100:.1f}%)')


## 4. Outlier Removal & Feature Scaling


In [ ]:
def detect_outliers_iqr(df, features, multiplier=3.0):
    """IQR outlier detection. Default multiplier=3.0 (relaxed from 1.5)."""
    mask = pd.Series(False, index=df.index)
    details = []
    for feat in features:
        Q1, Q3 = df[feat].quantile(0.25), df[feat].quantile(0.75)
        IQR = Q3 - Q1
        lo, hi = Q1 - multiplier * IQR, Q3 + multiplier * IQR
        feat_outliers = (df[feat] < lo) | (df[feat] > hi)
        mask = mask | feat_outliers
        details.append({'Feature': feat, 'Lower': round(lo, 6), 'Upper': round(hi, 6),
                        'N_Outliers': int(feat_outliers.sum()),
                        'Pct_Outliers': round(feat_outliers.mean() * 100, 2)})
    return mask, pd.DataFrame(details)

train_mask,    train_outlier_detail = detect_outliers_iqr(df_train,    FEATURE_COLS, multiplier=3.0)
validate_mask, _                   = detect_outliers_iqr(df_validate,  FEATURE_COLS, multiplier=3.0)
test_mask,     _                   = detect_outliers_iqr(df_test,      FEATURE_COLS, multiplier=3.0)

print('Outlier detail (Training, IQR × 3.0):')
print(train_outlier_detail.to_string(index=False))

summary = pd.DataFrame({
    'Split':       ['Train', 'Validate', 'Test'],
    'Original':    [len(df_train), len(df_validate), len(df_test)],
    'Outliers':    [int(train_mask.sum()), int(validate_mask.sum()), int(test_mask.sum())],
    'Remaining':   [int((~train_mask).sum()), int((~validate_mask).sum()), int((~test_mask).sum())],
    'Pct_Removed': [round(train_mask.mean()*100,1), round(validate_mask.mean()*100,1),
                    round(test_mask.mean()*100,1)],
})
print('\nOutlier Removal Summary (IQR × 3.0):')
print(summary.to_string(index=False))


In [ ]:
df_train_clean    = df_train[~train_mask].copy().reset_index(drop=True)
df_validate_clean = df_validate[~validate_mask].copy().reset_index(drop=True)
df_test_clean     = df_test[~test_mask].copy().reset_index(drop=True)

X_train_raw    = df_train_clean[FEATURE_COLS].values
X_validate_raw = df_validate_clean[FEATURE_COLS].values
X_test_raw     = df_test_clean[FEATURE_COLS].values

scaler = RobustScaler()
X_train_scaled    = scaler.fit_transform(X_train_raw)
X_validate_scaled = scaler.transform(X_validate_raw)
X_test_scaled     = scaler.transform(X_test_raw)

print('Feature matrices (raw + RobustScaled):')
print(f'  X_train_raw:    {X_train_raw.shape}')
print(f'  X_validate_raw: {X_validate_raw.shape}')
print(f'  X_test_raw:     {X_test_raw.shape}')

print('\nScaled training data stats (should be centred near 0):')
for i, feat in enumerate(FEATURE_COLS):
    iqr_val = np.percentile(X_train_scaled[:,i], 75) - np.percentile(X_train_scaled[:,i], 25)
    print(f'  {feat:25s}  median={np.median(X_train_scaled[:,i]):.3f}  IQR={iqr_val:.3f}')

# Note on chip_rate IQR=0 after scaling
chip_idx = FEATURE_COLS.index('chip_rate') if 'chip_rate' in FEATURE_COLS else -1
if chip_idx >= 0:
    chip_iqr = np.percentile(X_train_scaled[:,chip_idx], 75) - np.percentile(X_train_scaled[:,chip_idx], 25)
    if chip_iqr < 0.01:
        print('\n⚠️  chip_rate IQR≈0 after scaling: nearly all MCC 4121 merchants use chip-only.')
        print('   This feature adds little discriminative power in scaled space.')
        print('   Raw feature space will likely outperform scaled — this is expected.')


## 5. K-Means Clustering — Hyperparameter Tuning

In [ ]:
print('=' * 80)
print('K-MEANS HYPERPARAMETER TUNING')
print('=' * 80)

k_values       = list(range(2, 16))
init_methods   = ['k-means++', 'random']
n_init_values  = [1, 5, 10, 20]
kmeans_results = []

for k in k_values:
    for init_m in init_methods:
        for n_i in n_init_values:
            for data_name, X_tr, X_val in [('raw', X_train_raw, X_validate_raw),
                                            ('scaled', X_train_scaled, X_validate_scaled)]:
                km = KMeans(n_clusters=k, init=init_m, n_init=n_i, random_state=42)
                tr_labels  = km.fit_predict(X_tr)
                val_labels = km.predict(X_val)
                kmeans_results.append({
                    'k': k, 'init': init_m, 'n_init': n_i, 'data': data_name,
                    'train_silhouette':    silhouette_score(X_tr, tr_labels),
                    'train_db':            davies_bouldin_score(X_tr, tr_labels),
                    'train_inertia':       km.inertia_,
                    'validate_silhouette': silhouette_score(X_val, val_labels),
                    'validate_db':         davies_bouldin_score(X_val, val_labels),
                })

kmeans_df = pd.DataFrame(kmeans_results)
best_km   = kmeans_df.loc[kmeans_df['validate_silhouette'].idxmax()]
print(f'\nBest K-Means config (by Validation Silhouette):')
print(f'  k={int(best_km["k"])}, init="{best_km["init"]}", n_init={int(best_km["n_init"])}, data={best_km["data"]}')
print(f'  Train Silhouette: {best_km["train_silhouette"]:.4f}    Train DB: {best_km["train_db"]:.4f}')
print(f'  Val   Silhouette: {best_km["validate_silhouette"]:.4f}    Val   DB: {best_km["validate_db"]:.4f}')
print(f'\nTop 10 configurations:')
print(kmeans_df.nlargest(10, 'validate_silhouette')[
    ['k','init','n_init','data','train_silhouette','validate_silhouette','train_db','validate_db']
].to_string(index=False))


In [ ]:
for data_label in ['raw', 'scaled']:
    sub = kmeans_df[kmeans_df['data'] == data_label]
    best_per_k = sub.groupby('k').apply(lambda g: g.loc[g['validate_silhouette'].idxmax()]).reset_index(drop=True)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'K-Means Diagnostics — {data_label.upper()} data', fontsize=13, fontweight='bold')
    axes[0].plot(best_per_k['k'], best_per_k['train_inertia'], 'o-', color='steelblue', label='Train')
    axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
    axes[0].set_title('Elbow Plot (Train Inertia)'); axes[0].grid(True, alpha=0.3)
    axes[1].plot(best_per_k['k'], best_per_k['train_silhouette'],    'o-',  color='steelblue', label='Train')
    axes[1].plot(best_per_k['k'], best_per_k['validate_silhouette'], 's--', color='orange',    label='Validate')
    axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
    axes[1].set_title('Silhouette Score vs k'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
    axes[2].plot(best_per_k['k'], best_per_k['train_db'],    'o-',  color='steelblue', label='Train')
    axes[2].plot(best_per_k['k'], best_per_k['validate_db'], 's--', color='orange',    label='Validate')
    axes[2].set_xlabel('k'); axes[2].set_ylabel('Davies-Bouldin Index')
    axes[2].set_title('Davies-Bouldin vs k'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
best_k       = int(best_km['k'])
best_init    = best_km['init']
best_n_init  = int(best_km['n_init'])
best_km_data = best_km['data']

X_tr_km  = X_train_raw    if best_km_data == 'raw' else X_train_scaled
X_val_km = X_validate_raw if best_km_data == 'raw' else X_validate_scaled
X_te_km  = X_test_raw     if best_km_data == 'raw' else X_test_scaled

km_final      = KMeans(n_clusters=best_k, init=best_init, n_init=best_n_init, random_state=42)
km_tr_labels  = km_final.fit_predict(X_tr_km)
km_val_labels = km_final.predict(X_val_km)
km_te_labels  = km_final.predict(X_te_km)

train_silhouette_km    = silhouette_score(X_tr_km,  km_tr_labels)
train_db_km            = davies_bouldin_score(X_tr_km,  km_tr_labels)
validate_silhouette_km = silhouette_score(X_val_km, km_val_labels)
validate_db_km         = davies_bouldin_score(X_val_km, km_val_labels)
test_silhouette_km     = silhouette_score(X_te_km,  km_te_labels)
test_db_km             = davies_bouldin_score(X_te_km,  km_te_labels)

print(f'Best K-Means Final Results (k={best_k}, {best_km_data}):')
print(f'  Train:    Silhouette={train_silhouette_km:.4f}  DB={train_db_km:.4f}')
print(f'  Validate: Silhouette={validate_silhouette_km:.4f}  DB={validate_db_km:.4f}')
print(f'  Test:     Silhouette={test_silhouette_km:.4f}  DB={test_db_km:.4f}')

unique, counts = np.unique(km_tr_labels, return_counts=True)
print(f'\nTrain cluster sizes: {dict(zip(unique.tolist(), counts.tolist()))}')


In [ ]:
df_train_clust    = df_train_clean.copy()
df_train_clust['cluster'] = km_tr_labels
cluster_profiles = df_train_clust.groupby('cluster')[FEATURE_COLS].mean()
print('K-Means Cluster Mean Feature Profiles:')
print(cluster_profiles.T.to_string())

fig, ax = plt.subplots(figsize=(12, 6))
mins = cluster_profiles.min(); maxs = cluster_profiles.max()
cluster_profiles_norm = (cluster_profiles - mins) / (maxs - mins + 1e-12)
sns.heatmap(cluster_profiles_norm.T, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax)
ax.set_title(f'K-Means Cluster Profiles (Normalised Means) — k={best_k}', fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster'); ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()


## 6. Hierarchical Clustering — Hyperparameter Tuning

In [ ]:
sample_size = min(300, len(X_train_scaled))
np.random.seed(42)
sample_idx = np.random.choice(len(X_train_scaled), sample_size, replace=False)
X_sample   = X_train_scaled[sample_idx]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(f'Dendrograms (Sample n={sample_size}) — Scaled Data', fontsize=14, fontweight='bold')
for ax, method in zip(axes.flatten(), ['ward', 'complete', 'average', 'single']):
    Z = linkage(X_sample, method=method)
    dendrogram(Z, ax=ax, truncate_mode='lastp', p=30, leaf_rotation=45, leaf_font_size=8)
    ax.set_title(f'{method.capitalize()} linkage', fontweight='bold')
    ax.set_xlabel('Sample index (or cluster size)')
    ax.set_ylabel('Distance')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
print('=' * 80)
print('HIERARCHICAL CLUSTERING HYPERPARAMETER TUNING')
print('=' * 80)

linkage_methods  = ['ward', 'complete', 'average', 'single']
n_clusters_range = list(range(2, 11))
hier_results     = []

for data_name, X_tr, X_val in [('raw', X_train_raw, X_validate_raw),
                                 ('scaled', X_train_scaled, X_validate_scaled)]:
    for link_m in linkage_methods:
        print(f'  Computing {link_m} linkage on {data_name} data...')
        Z = linkage(X_tr, method=link_m)
        for n_cl in n_clusters_range:
            tr_labels = fcluster(Z, n_cl, criterion='maxclust')
            centres   = np.array([X_tr[tr_labels == c].mean(axis=0)
                                   for c in range(1, n_cl+1)
                                   if (tr_labels == c).sum() > 0])
            if len(centres) < 2:
                continue
            val_labels = cdist(X_val, centres).argmin(axis=1)
            if len(np.unique(val_labels)) < 2:
                continue
            hier_results.append({
                'linkage': link_m, 'n_clusters': n_cl, 'data': data_name,
                'train_silhouette':    silhouette_score(X_tr,  tr_labels),
                'train_db':            davies_bouldin_score(X_tr,  tr_labels),
                'validate_silhouette': silhouette_score(X_val, val_labels),
                'validate_db':         davies_bouldin_score(X_val, val_labels),
            })

hier_df   = pd.DataFrame(hier_results)
best_hier = hier_df.loc[hier_df['validate_silhouette'].idxmax()]
print(f'\nBest Hierarchical config:')
print(f'  linkage={best_hier["linkage"]}, n_clusters={int(best_hier["n_clusters"])}, data={best_hier["data"]}')
print(f'  Train: Sil={best_hier["train_silhouette"]:.4f}  DB={best_hier["train_db"]:.4f}')
print(f'  Val:   Sil={best_hier["validate_silhouette"]:.4f}  DB={best_hier["validate_db"]:.4f}')
print(f'\nTop 10:')
print(hier_df.nlargest(10, 'validate_silhouette')[
    ['linkage','n_clusters','data','train_silhouette','validate_silhouette','train_db','validate_db']
].to_string(index=False))


In [ ]:
best_hier_link = best_hier['linkage']
best_hier_nc   = int(best_hier['n_clusters'])
best_hier_data = best_hier['data']

X_tr_h  = X_train_raw    if best_hier_data == 'raw' else X_train_scaled
X_val_h = X_validate_raw if best_hier_data == 'raw' else X_validate_scaled
X_te_h  = X_test_raw     if best_hier_data == 'raw' else X_test_scaled

Z_final       = linkage(X_tr_h, method=best_hier_link)
hier_tr_labels = fcluster(Z_final, best_hier_nc, criterion='maxclust')
centres_final  = np.array([X_tr_h[hier_tr_labels == c].mean(axis=0)
                             for c in range(1, best_hier_nc+1)])
hier_val_labels = cdist(X_val_h, centres_final).argmin(axis=1)
hier_te_labels  = cdist(X_te_h,  centres_final).argmin(axis=1)

train_silhouette_hier    = silhouette_score(X_tr_h,  hier_tr_labels)
train_db_hier            = davies_bouldin_score(X_tr_h,  hier_tr_labels)
validate_silhouette_hier = silhouette_score(X_val_h, hier_val_labels)
validate_db_hier         = davies_bouldin_score(X_val_h, hier_val_labels)
test_silhouette_hier     = silhouette_score(X_te_h,  hier_te_labels)
test_db_hier             = davies_bouldin_score(X_te_h,  hier_te_labels)

print(f'Best Hierarchical Final Results (linkage={best_hier_link}, n={best_hier_nc}, {best_hier_data}):')
print(f'  Train:    Silhouette={train_silhouette_hier:.4f}  DB={train_db_hier:.4f}')
print(f'  Validate: Silhouette={validate_silhouette_hier:.4f}  DB={validate_db_hier:.4f}')
print(f'  Test:     Silhouette={test_silhouette_hier:.4f}  DB={test_db_hier:.4f}')


## 7. DBSCAN — Epsilon Tuning


In [ ]:
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_train_scaled)
distances, _ = nn.kneighbors(X_train_scaled)
knn_dist = np.sort(distances[:, -1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(knn_dist, color='steelblue', linewidth=1.5)
ax.set_xlabel('Sorted Points')
ax.set_ylabel('5th Nearest-Neighbour Distance')
ax.set_title('KNN Distance Plot — Epsilon Selection for DBSCAN (Scaled Data)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

knee_approx = np.percentile(knn_dist, 90)
print(f'Suggested epsilon starting point (90th percentile): {knee_approx:.4f}')


In [ ]:
print('=' * 80)
print('DBSCAN HYPERPARAMETER TUNING (Scaled Data)')
print('=' * 80)

eps_values         = [round(knee_approx * m, 4) for m in [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 2.5]]
min_samples_values = [3, 5, 10, 15]
dbscan_results     = []

for eps in eps_values:
    for ms in min_samples_values:
        db = DBSCAN(eps=eps, min_samples=ms)
        tr_labels  = db.fit_predict(X_train_scaled)
        n_clusters = len(set(tr_labels)) - (1 if -1 in tr_labels else 0)
        noise_pct  = (tr_labels == -1).mean() * 100
        if n_clusters < 2:
            continue
        non_noise  = tr_labels != -1
        if non_noise.sum() < 10:
            continue

        # Derive centroids from non-noise training points
        centres_db = np.array([X_train_scaled[non_noise][tr_labels[non_noise] == c].mean(axis=0)
                                for c in range(n_clusters)
                                if (tr_labels[non_noise] == c).sum() > 0])
        if len(centres_db) < 2:
            continue

        # ── IMPROVEMENT: consistent val/test assignment ──────────────────
        val_labels = cdist(X_validate_scaled, centres_db).argmin(axis=1)
        te_labels  = cdist(X_test_scaled,     centres_db).argmin(axis=1)
        if len(np.unique(val_labels)) < 2 or len(np.unique(te_labels)) < 2:
            continue

        dbscan_results.append({
            'eps': eps, 'min_samples': ms, 'n_clusters': n_clusters,
            'noise_pct': round(noise_pct, 2),
            # Training: non-noise only (DBSCAN-native)
            'train_silhouette': silhouette_score(X_train_scaled[non_noise], tr_labels[non_noise]),
            'train_db':         davies_bouldin_score(X_train_scaled[non_noise], tr_labels[non_noise]),
            # Validate/Test: all points (noise forced to nearest centroid)
            'validate_silhouette': silhouette_score(X_validate_scaled, val_labels),
            'validate_db':         davies_bouldin_score(X_validate_scaled, val_labels),
            'test_silhouette':     silhouette_score(X_test_scaled, te_labels),
            'test_db':             davies_bouldin_score(X_test_scaled, te_labels),
        })

dbscan_df = pd.DataFrame(dbscan_results)
if len(dbscan_df) > 0:
    best_dbscan = dbscan_df.loc[dbscan_df['validate_silhouette'].idxmax()]
    print(f'\nBest DBSCAN config:')
    print(f'  eps={best_dbscan["eps"]}, min_samples={int(best_dbscan["min_samples"])}, '
          f'n_clusters={int(best_dbscan["n_clusters"])}, noise={best_dbscan["noise_pct"]:.1f}%')
    print(f'  Train (non-noise): Sil={best_dbscan["train_silhouette"]:.4f}  DB={best_dbscan["train_db"]:.4f}')
    print(f'  Val  (all points): Sil={best_dbscan["validate_silhouette"]:.4f}  DB={best_dbscan["validate_db"]:.4f}')
    print(f'  Test (all points): Sil={best_dbscan["test_silhouette"]:.4f}  DB={best_dbscan["test_db"]:.4f}')
    print(f'\nTop 10:')
    print(dbscan_df.nlargest(10, 'validate_silhouette').to_string(index=False))
    train_silhouette_dbscan    = best_dbscan['train_silhouette']
    train_db_dbscan            = best_dbscan['train_db']
    validate_silhouette_dbscan = best_dbscan['validate_silhouette']
    validate_db_dbscan         = best_dbscan['validate_db']
    test_silhouette_dbscan     = best_dbscan['test_silhouette']
    test_db_dbscan             = best_dbscan['test_db']
else:
    print('No valid DBSCAN configurations found.')
    train_silhouette_dbscan = validate_silhouette_dbscan = test_silhouette_dbscan = np.nan
    train_db_dbscan = validate_db_dbscan = test_db_dbscan = np.nan


## 8. Gaussian Mixture Model (GMM) — Hyperparameter Tuning

GMM is a soft/probabilistic clustering method that models data as a mixture of
Gaussian distributions. Every merchant gets a probability of belonging to each cluster.

### Improvement: BIC/AIC used as secondary selection criterion alongside silhouette


In [ ]:
print('=' * 80)
print('GMM HYPERPARAMETER TUNING')
print('=' * 80)

n_components_range = list(range(2, 12))
covariance_types   = ['full', 'tied', 'diag', 'spherical']
gmm_results        = []

for data_name, X_tr, X_val in [('raw', X_train_raw, X_validate_raw),
                                 ('scaled', X_train_scaled, X_validate_scaled)]:
    for cov_type in covariance_types:
        for n_comp in n_components_range:
            try:
                gmm = GaussianMixture(n_components=n_comp, covariance_type=cov_type,
                                      random_state=42, max_iter=200, n_init=5)
                gmm.fit(X_tr)
                tr_labels  = gmm.predict(X_tr)
                val_labels = gmm.predict(X_val)
                if len(np.unique(tr_labels)) < 2 or len(np.unique(val_labels)) < 2:
                    continue
                gmm_results.append({
                    'n_components': n_comp, 'covariance_type': cov_type, 'data': data_name,
                    'bic':  gmm.bic(X_tr),
                    'aic':  gmm.aic(X_tr),
                    'train_silhouette':    silhouette_score(X_tr,  tr_labels),
                    'train_db':            davies_bouldin_score(X_tr,  tr_labels),
                    'validate_silhouette': silhouette_score(X_val, val_labels),
                    'validate_db':         davies_bouldin_score(X_val, val_labels),
                })
            except Exception:
                continue

gmm_df   = pd.DataFrame(gmm_results)
best_gmm = gmm_df.loc[gmm_df['validate_silhouette'].idxmax()]

# ── IMPROVEMENT: also show BIC-optimal model ────────────────────────────────
raw_gmm = gmm_df[gmm_df['data'] == 'raw']
bic_best = raw_gmm.loc[raw_gmm['bic'].idxmin()] if not raw_gmm.empty else best_gmm

print(f'\nBest GMM config (by Validation Silhouette):')
print(f'  n_components={int(best_gmm["n_components"])}, cov_type={best_gmm["covariance_type"]}, data={best_gmm["data"]}')
print(f'  BIC={best_gmm["bic"]:.1f}  AIC={best_gmm["aic"]:.1f}')
print(f'  Train: Sil={best_gmm["train_silhouette"]:.4f}  DB={best_gmm["train_db"]:.4f}')
print(f'  Val:   Sil={best_gmm["validate_silhouette"]:.4f}  DB={best_gmm["validate_db"]:.4f}')
print(f'\nBest GMM config (by BIC — raw data):')
print(f'  n_components={int(bic_best["n_components"])}, cov_type={bic_best["covariance_type"]}')
print(f'  BIC={bic_best["bic"]:.1f}  AIC={bic_best["aic"]:.1f}')
print(f'  Train: Sil={bic_best["train_silhouette"]:.4f}  DB={bic_best["train_db"]:.4f}')
print(f'  Val:   Sil={bic_best["validate_silhouette"]:.4f}  DB={bic_best["validate_db"]:.4f}')
print(f'\nTop 10 (by Validation Silhouette):')
print(gmm_df.nlargest(10, 'validate_silhouette')[
    ['n_components','covariance_type','data','bic','train_silhouette','validate_silhouette','train_db','validate_db']
].to_string(index=False))


In [ ]:
for cov in ['full', 'tied', 'diag', 'spherical']:
    for dat in ['raw', 'scaled']:
        sub = gmm_df[(gmm_df['covariance_type'] == cov) & (gmm_df['data'] == dat)]
        if sub.empty:
            continue
        sub = sub.groupby('n_components')[['bic','aic','validate_silhouette']].mean().reset_index()
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle(f'GMM — covariance_type="{cov}", data={dat}', fontsize=12, fontweight='bold')
        axes[0].plot(sub['n_components'], sub['bic'], 'o-', label='BIC', color='steelblue')
        axes[0].plot(sub['n_components'], sub['aic'], 's--', label='AIC', color='orange')
        axes[0].set_xlabel('n_components'); axes[0].set_ylabel('Score')
        axes[0].set_title('BIC / AIC (lower is better)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
        axes[1].plot(sub['n_components'], sub['validate_silhouette'], 'o-', color='green')
        axes[1].set_xlabel('n_components'); axes[1].set_ylabel('Silhouette Score')
        axes[1].set_title('Validation Silhouette'); axes[1].grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


In [ ]:
best_gmm_nc   = int(best_gmm['n_components'])
best_gmm_cov  = best_gmm['covariance_type']
best_gmm_data = best_gmm['data']

X_tr_gm  = X_train_raw    if best_gmm_data == 'raw' else X_train_scaled
X_val_gm = X_validate_raw if best_gmm_data == 'raw' else X_validate_scaled
X_te_gm  = X_test_raw     if best_gmm_data == 'raw' else X_test_scaled

gmm_final     = GaussianMixture(n_components=best_gmm_nc, covariance_type=best_gmm_cov,
                                 random_state=42, max_iter=200, n_init=5)
gmm_final.fit(X_tr_gm)
gmm_tr_labels  = gmm_final.predict(X_tr_gm)
gmm_val_labels = gmm_final.predict(X_val_gm)
gmm_te_labels  = gmm_final.predict(X_te_gm)

train_silhouette_gmm    = silhouette_score(X_tr_gm,  gmm_tr_labels)
train_db_gmm            = davies_bouldin_score(X_tr_gm,  gmm_tr_labels)
validate_silhouette_gmm = silhouette_score(X_val_gm, gmm_val_labels)
validate_db_gmm         = davies_bouldin_score(X_val_gm, gmm_val_labels)
test_silhouette_gmm     = silhouette_score(X_te_gm,  gmm_te_labels)
test_db_gmm             = davies_bouldin_score(X_te_gm,  gmm_te_labels)

print(f'Best GMM Final Results (n={best_gmm_nc}, cov={best_gmm_cov}, {best_gmm_data}):')
print(f'  Train:    Silhouette={train_silhouette_gmm:.4f}  DB={train_db_gmm:.4f}')
print(f'  Validate: Silhouette={validate_silhouette_gmm:.4f}  DB={validate_db_gmm:.4f}')
print(f'  Test:     Silhouette={test_silhouette_gmm:.4f}  DB={test_db_gmm:.4f}')

probs     = gmm_final.predict_proba(X_tr_gm)
max_probs = probs.max(axis=1)
print(f'\nCluster assignment confidence (max probability):')
print(f'  Mean:            {max_probs.mean():.3f}')
print(f'  Median:          {np.median(max_probs):.3f}')
print(f'  <50% confidence: {(max_probs < 0.5).sum()} merchants')
print(f'  >90% confidence: {(max_probs > 0.9).sum()} merchants')


## 9. Model Comparison & Best Model Selection

In [ ]:
results = pd.DataFrame([
    {'Model': f'K-Means (k={best_k})',
     'Train Sil': train_silhouette_km,    'Val Sil': validate_silhouette_km,    'Test Sil': test_silhouette_km,
     'Train DB':  train_db_km,            'Val DB':  validate_db_km,            'Test DB':  test_db_km},
    {'Model': f'Hierarchical ({best_hier_link}, n={best_hier_nc})',
     'Train Sil': train_silhouette_hier,  'Val Sil': validate_silhouette_hier,  'Test Sil': test_silhouette_hier,
     'Train DB':  train_db_hier,          'Val DB':  validate_db_hier,          'Test DB':  test_db_hier},
    {'Model': f'DBSCAN (eps={best_dbscan["eps"] if len(dbscan_df) > 0 else "N/A"})',
     'Train Sil': train_silhouette_dbscan, 'Val Sil': validate_silhouette_dbscan, 'Test Sil': test_silhouette_dbscan,
     'Train DB':  train_db_dbscan,         'Val DB':  validate_db_dbscan,         'Test DB':  test_db_dbscan},
    {'Model': f'GMM (n={best_gmm_nc}, {best_gmm_cov})',
     'Train Sil': train_silhouette_gmm,   'Val Sil': validate_silhouette_gmm,   'Test Sil': test_silhouette_gmm,
     'Train DB':  train_db_gmm,           'Val DB':  validate_db_gmm,           'Test DB':  test_db_gmm},
])

float_cols = ['Train Sil','Val Sil','Test Sil','Train DB','Val DB','Test DB']
print('=' * 80)
print('MODEL COMPARISON SUMMARY')
print('=' * 80)
print(results.to_string(index=False, float_format='{:.4f}'.format))

# Highlight best
best_idx = results['Test Sil'].idxmax()
print(f'\n✓ Best model by Test Silhouette: {results.loc[best_idx, "Model"]}')
best_db_idx = results['Test DB'].idxmin()
print(f'✓ Best model by Test Davies-Bouldin: {results.loc[best_db_idx, "Model"]}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
models = results['Model'].tolist()
x = np.arange(len(models))
w = 0.25

for ax, (sil_train, sil_val, sil_test), title in zip(
    axes,
    [('Train Sil', 'Val Sil', 'Test Sil'), ('Train DB', 'Val DB', 'Test DB')],
    ['Silhouette Score (higher = better)', 'Davies-Bouldin Index (lower = better)']
):
    ax.bar(x - w, results[sil_train], w, label='Train',    alpha=0.8)
    ax.bar(x,     results[sil_val],   w, label='Validate', alpha=0.8)
    ax.bar(x + w, results[sil_test],  w, label='Test',     alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=15, ha='right', fontsize=9)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Clustering Model Comparison — MCC 4121', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 10. Final Model — Cluster Profiles & Business Interpretation

In [ ]:
# Use the best model (GMM) for final cluster profiling
df_final = df_train_clean.copy()
df_final['cluster'] = gmm_tr_labels
df_final['cluster_prob'] = gmm_final.predict_proba(X_tr_gm).max(axis=1)

profile = df_final.groupby('cluster')[FEATURE_COLS].mean()
sizes   = df_final.groupby('cluster').size().rename('n_merchants')
profile = profile.join(sizes)

print('Final Cluster Profiles (GMM — Train):')
print(profile.T.to_string())

# Normalised heatmap
fig, ax = plt.subplots(figsize=(12, 6))
mins = profile[FEATURE_COLS].min(); maxs = profile[FEATURE_COLS].max()
profile_norm = (profile[FEATURE_COLS] - mins) / (maxs - mins + 1e-12)
sns.heatmap(profile_norm.T, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Final GMM Cluster Profiles (Normalised Means) — MCC 4121', fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster'); ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

print('\nBusiness Interpretation:')
for cl in sorted(df_final['cluster'].unique()):
    row = profile.loc[cl]
    print(f'\n  Cluster {cl} ({int(row["n_merchants"])} merchants):')
    print(f'    avg_txn_amount = {row["avg_txn_amount"]:.2f}   cost_percent = {row["cost_percent"]:.4f}')
    print(f'    chip_rate = {row["chip_rate"]:.2f}   visa_rate = {row["visa_rate"]:.2f}   credit_rate = {row["credit_rate"]:.2f}')


In [ ]:
# Uncertainty distribution — merchants close to decision boundary
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_final['cluster_prob'], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(0.9, color='red',    linestyle='--', label='>90% confident')
ax.axvline(0.5, color='orange', linestyle='--', label='>50% confident')
ax.set_xlabel('Max Cluster Membership Probability')
ax.set_ylabel('Number of Merchants')
ax.set_title('GMM Assignment Confidence — Training Merchants', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

uncertain = df_final[df_final['cluster_prob'] < 0.8]
print(f'Merchants with <80% assignment confidence: {len(uncertain)} of {len(df_final)}')
if len(uncertain) > 0:
    print(uncertain[FEATURE_COLS + ['cluster','cluster_prob']].to_string())
